In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os

GESTURE = "YOU"   # CHANGE THIS
SAMPLES = 50

SAVE_DIR = f"dataset/{GESTURE}"
os.makedirs(SAVE_DIR, exist_ok=True)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2)
cap = cv2.VideoCapture(0)

count = 0
print("Press Q to quit")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    landmarks = []

    if result.multi_hand_landmarks:
        for hand in result.multi_hand_landmarks:
            for lm in hand.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

    if len(landmarks) in [63, 126]:
        np.save(f"{SAVE_DIR}/{count}.npy", np.array(landmarks))
        count += 1
        print(f"Saved {count}")

    cv2.imshow("Collecting Data", frame)
    if cv2.waitKey(1) & 0xFF == ord('q') or count >= SAMPLES:
        break

cap.release()
cv2.destroyAllWindows()

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

DATASET_DIR = "Train_words"
SAVE_DIR = "processed_data"
os.makedirs(SAVE_DIR, exist_ok=True)

for label in os.listdir(DATASET_DIR):
    label_path = os.path.join(DATASET_DIR, label)
    save_label_path = os.path.join(SAVE_DIR, label)
    os.makedirs(save_label_path, exist_ok=True)

    for img_name in os.listdir(label_path):
        img_path = os.path.join(label_path, img_name)
        image = cv2.imread(img_path)

        if image is None:
            continue

        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        landmarks = []

        if result.multi_hand_landmarks:
            for hand in result.multi_hand_landmarks:
                for lm in hand.landmark:
                    landmarks.extend([lm.x, lm.y, lm.z])

            if len(landmarks) == 63:
                np.save(
                    os.path.join(save_label_path, img_name.replace(".jpg", ".npy")),
                    np.array(landmarks)
                )

print("✅ Landmark extraction complete")

✅ Landmark extraction complete
